# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [4]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [5]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [6]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

Let's look at an example document to see if everything worked as expected!

In [7]:
loan_complaint_data[0]

Document(metadata={'source': './data/complaints.csv', 'row': 0, 'Date received': '03/27/25', 'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with your lender or servicer', 'Sub-issue': 'Trouble with how payments are being handled', 'Consumer complaint narrative': "The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers.", 'Company public response': 'None', 'Company'

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [8]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [9]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [10]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [11]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [12]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [13]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the complaints provided, appears to be dealing with the lender or servicer, including problems such as errors in loan balances, misapplied payments, wrongful denials of payment plans, and issues related to loan management and information accuracy. Many complaints also involve issues like incorrect reporting of account status, problems with how payments are applied, and mismanagement resulting in increased balances or adverse credit impacts.'

In [14]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, yes, some complaints were not handled in a timely manner. Specifically, the complaint involving the company MOHELA received a "No" response for the "Timely response?" criterion, indicating it was not handled promptly. Additionally, multiple complaints related to delays and lack of responses, such as those about account issues, loan processing delays, and unresponded complaints sent to Aidvantage and other servicers, suggest delays in handling complaints.\n\nIn summary, at least one complaint explicitly was not handled in a timely manner, and there are indications of delays in others as well.'

In [15]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans primarily due to a variety of factors outlined in the complaints, including:\n\n1. **Lack of Clear Communication and Notification:** Many borrowers were not adequately informed when their loans resumed or when loan servicers changed hands, leading to unintentional delinquency. For example, some were unaware of payment due dates or loan transfers, and late payments were reported without prior notice.\n\n2. **Problems with Payment Processing and Application:** Borrowers reported difficulty in applying extra payments toward principal, with some payments being automatically applied to interest rather than reducing the loan balance, making it harder to pay off loans quickly.\n\n3. **Accumulation of Interest and Unmanageable Debt:** Several complaints highlighted that interest continued to accrue during periods of forbearance or deferment, often negating any payments made, which extended the repayment timeline and increased total debt.\n\n4. **Insuffici

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [16]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [17]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [18]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with student loans appears to involve problems with dealing with lenders or servicers, such as disputes over fees, issues with repayment procedures, providing inaccurate information, or difficulties in managing loan aspects like applying additional funds or understanding loan balances. Specifically, complaints frequently mention:\n\n- Disputes over fees and charges\n- Challenges in making repayments or applying payments correctly\n- Receiving incorrect or bad information about loans\n- Issues with loan terms and balances\n- Lack of transparency or trustworthiness from loan servicers\n\nTherefore, the most common issue seems to be complications arising from dealing with the loan servicers or lenders, including miscommunication, errors, and disputes related to fees and repayment details.'

In [19]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided context, all of the complaints listed indicate that the companies responded in a timely manner. Specifically, the responses to complaints about incorrect information on reports and issues with collection activity are marked as "Timely response?": "Yes." Therefore, there is no evidence in the provided data to suggest that any complaints did not get handled in a timely manner.'

In [20]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans for various reasons, including issues with loan management and communication problems. Some common reasons based on the provided context are:\n\n1. Problems with payment plans and forbearances, where borrowers were steered into the wrong types of forbearances or were not properly informed about their options, leading to unpaid or missed payments.\n2. Mismanagement or lack of response from loan servicers, such as failing to respond to applications for deferment or forbearance, resulting in continued billing and collection efforts despite borrowers' efforts to qualify for relief.\n3. Technical issues with payments, such as payments being reversed repeatedly due to errors or incorrect bank information, and servicers blaming the borrowers while failing to resolve the issues.\n4. Lack of communication or miscommunication from loan servicers, such as not informing borrowers about loan transfers, missing emails, or failing to notify borrowers of changes 

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer:

A good example of a query where BM25 outperforms embeddings is: “Tell me about Sweet v. Cardona.”

This query contains specific, uncommon terms: proper nouns from a legal case. BM25 excels at retrieving documents with exact keyword matches, so it will prioritize documents that explicitly contain “Sweet” and “Cardona.”

In contrast, semantic search with embeddings might not recognize these as meaningful beyond being rare words. It could return semantically related legal cases, but miss documents about this specific one, especially if the case isn’t prominent enough to have distinct semantic representations.

Therefore, BM25 is more reliable for queries that depend heavily on rare or specific terms.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [21]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [22]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [23]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans, particularly student loans, appears to be problems related to dealing with lenders or servicers. These issues include receiving bad or incorrect information about the loan, errors in loan balances, misapplied payments, wrongful denials of payment plans, and mishandling or mismanagement of loan data. Many complaints also involve lack of communication, unapproved transfers of loans without proper notification, and disputes over inaccurate account information.'

In [24]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, yes, there are complaints that did not get handled in a timely manner. Specifically, one complaint regarding a student loan issue has been open for over a year without resolution, despite ongoing follow-up and requests for review and adjustments. The complainant stated that it has been nearly 18 months with no resolution and that they are still awaiting responses.'

In [25]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for several reasons, including:\n\n1. Lack of awareness or understanding: Some borrowers were not aware they needed to repay their financial aid or loan, often because they were not informed properly by financial aid officers.\n2. Poor communication from lenders or servicers: Borrowers reported not receiving timely notices, notifications about payment due dates, or information when loans were transferred between companies, which hindered their ability to make payments.\n3. Difficulty managing interest accumulation: For many, interest continued to grow even when they made payments, especially if they chose options like forbearance or deferment, which allowed interest to accrue and increase the total debt.\n4. Financial hardship and affordability: Some borrowers could not afford increased payments or higher monthly obligations, which extended the repayment period and increased total interest paid.\n5. Administrative issues and errors: Discrepancies 

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [26]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [27]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [28]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issues with loans, based on the complaints provided, include:\n\n- Errors and inaccuracies in loan balances and interest calculations\n- Mismanagement of loan servicing, including improper handling of payments and loan transfer notifications\n- Unauthorized or unverified loan changes and reporting\n- Failure to provide necessary documentation such as original promissory notes\n- Problems with repayment plans, including difficulty in applying payments correctly and being steered into inappropriate options\n- Unlawful or suspicious loan modification, interest rate changes, or undisclosed terms\n- Mistaken or fraudulent reporting on credit reports\n- Lack of transparency and poor communication from loan servicers\n- Stress, harassment, and privacy violations related to loan collection practices\n\nWhile multiple issues are evident, errors in loan balances, interest miscalculations, and mishandling of payments and account information appear to be particularly prevalent and

In [29]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided context, several complaints indicate delays or failures in handling issues in a timely manner. For example:\n\n- The complaint received on 03/28/25 regarding a student loan application from MOHELA mentions that the consumer was told it would take 15 days for someone to contact them, but as of the date of the complaint, they had not heard back. The company response was "Closed with explanation" and "No," indicating response was not timely.\n- Multiple complaints, such as those received on 04/01/25, 04/12/25, 04/14/25, and later dates, include references to delays, unresponsiveness, or ongoing issues despite repeated follow-ups, with some cases noting no response after over a year.\n- Several complaints explicitly mention that responses or corrections were not handled in a timely manner, including instances where the company response was "Closed with explanation" and the "Timely response?" field was marked "No" or "Yes" depending on the case.\n\nTherefore, yes, the

In [30]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to issues such as mismanagement and errors by loan servicers, lack of proper communication and guidance, unawareness of available repayment options like income-driven plans, and systemic failures in the servicing process. Specific reasons include:\n\n- Receiving bad or incorrect information about their loans and repayment options.\n- Being steered into long-term forbearances without being informed of alternatives like income-driven repayment or rehabilitation programs, leading to increased interest and ballooning balances.\n- Servicers not following regulations regarding timely notices and proper collection practices.\n- Errors in loan balances, misapplied payments, wrongful reporting to credit bureaus, and inadequate updates or disclosures.\n- Systemic issues such as transfers of loans without proper notification, poor record retention, and lack of transparency, which have resulted in confusion, unintentional delinquency, and negati

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer:

Generating multiple reformulations of a user query can improve recall by increasing the chances of retrieving relevant documents that might otherwise be missed due to phrasing differences, typos, or incomplete input.

For example, a user may write:

> Wat do cant pay loan

This is a fairly extreme example, but even so, an LLM would likely be able to understand the users intent and rewrite it to something like:

> What can I do about being unable to repay my student loan?

Even when the original query is well-formed, reformulating it into multiple phrasings can still enhance retrieval. For example, starting with:

> What can I do about being unable to repay my student loan?

We can rewrite this to potentially retrieve better documents that would be missed by some of the queries:

> 1. How can I manage my student loan if I can’t afford to repay it?

> 2. What options are available if I'm struggling to pay back my student loans?

> 3. Is there any help for borrowers who can't make their student loan payments?

These three queries are all looking for help on the same topic, but their phrasing differences will help us to ensure we get all of the potentially helpful documents.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [31]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [32]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [33]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [34]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [35]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [36]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided context, appears to be problems related to federal student loan servicing, including errors in loan balances, misapplied payments, wrongful denials of payment plans, and issues with inaccurate or misleading credit reporting. Many complaints involve systemic breakdowns, such as incorrect information on credit reports, unfair interest rate increases, and failure of servicers to verify debt legitimacy or provide accurate account information.'

In [37]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, any complaints labeled with \'Timely response?\' as \'No\' indicate they were not handled in a timely manner. Specifically:\n\n- Complaint ID: 12709087 (row 441) - "Dealing with your lender or servicer" related to a student loan with MOHELA, received on 03/28/25. It was not handled timely.\n- Complaint ID: 12935889 (row 84) - Similar issue with MOHELA, received on 04/11/25. It was also not handled timely.\n\nThese complaints explicitly mention that the responses were not timely. Therefore, yes, some complaints did not get handled in a timely manner.'

In [38]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans primarily due to a variety of issues such as financial hardship, lack of proper information, mismanagement, and unforeseen circumstances. For instance, some borrowers experienced severe financial difficulties after graduation and relied on deferments or forbearance, which increased the total debt due to accrued interest. Others faced problems with loan servicing, such as being notified late about repayment obligations, or having payments resumed before the grace period ended. Additionally, some borrowers were misled about the value of their education and the manageability of their loans, and others encountered issues related to school closures, misrepresentation, and the inability to secure employment, making repayment impossible. If you're interested in a specific case or aspect, I can provide more details."

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [39]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [40]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [41]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be dealing with the servicing and administration of student loans, including errors in loan balances, misapplied payments, wrongful denials of repayment plans, and mishandling of loan documentation. Many complaints highlight problems such as receiving bad information about loans, incorrect reporting to credit bureaus, lack of transparency, and improper or unauthorized changes to loan terms. Servicing failures, including failure to provide proper documentation, improper transfer of loans without borrower notification, and inaccurate or incomplete information about loan status and balances, are prevalent issues.\n\nIn summary, the most common issue is **problems related to the handling and servicing of student loans**, especially errors and mismanagement by loan servicers that adversely affect borrowers’ credit, payments, and understanding of their loan obligations.'

In [42]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints data, several complaints indicate that issues were not handled in a timely manner. For example:\n\n- Complaint ID 12935889 (row 651): The complaint was filed about a month before the response, and the company responded "No" to timely response, indicating it was overdue. The complaint was marked "No" under "Timely response?" indicating it was not handled promptly.\n\n- Complaint ID 12935889 (row 89): The response was marked "No" under "Timely response?" and explicitly states the response was not timely.\n\nAdditionally, multiple complaints mention delays, long wait times, or failure to respond within expected timeframes, such as:\n\n- Complaint ID 12935889: The response was "No" for timely response.\n- Complaint ID 13062402: The complaint was responded to within the expected timeframe (marked "Yes" for timely response).\n- Complaint ID 12973003: Also responded timely.\n- Complaint ID 13410623: Marked "Yes" for timely response but still experienced delay

In [43]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including:\n\n1. Lack of proper notification and communication from loan servicers about payment due dates, account transfers, or delinquency status, leading to unawareness of repayment obligations.\n2. Mismanagement or incorrect handling of their accounts, such as errors in reporting, inaccurate balances, or inability to access or verify account information.\n3. Being steered into long-term forbearances or other repayment options that cause interest to accumulate substantially, making repayment more difficult over time.\n4. Financial hardships, such as unemployment, unexpected expenses, or personal crises, which hinder their ability to make payments.\n5. Unfavorable loan terms, like high interest rates and inability to qualify for income-driven repayment plans or loan forgiveness programs.\n6. Lack of transparency about loan details, interest accrual, or available repayment options, leading borrowers to underestimate their ob

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [44]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [45]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [46]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [47]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [48]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [49]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be difficulties and conflicts related to loan servicing and information accuracy. Specific recurring problems include:\n\n- Troubles with repayment and payment plans (e.g., issues with re-amortization, auto-debit setup, and payment amounts)\n- Disputes over loan status and incorrect reporting (e.g., loans in default when not, wrong account statuses)\n- Problems with loan information transparency and communication (e.g., lack of clarity about loan servicer changes, delays, or miscommunication)\n- Unauthorized or illegal reporting and breach of privacy\n- Troubles with loan forgiveness, cancellation, or discharge processes\n\nThese issues often involve mismanagement, inaccurate or inconsistent information, and lack of proper communication from loan servicers.\n\nIf you have a specific aspect you’re interested in, please let me know!'

In [50]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, it appears that several complaints were handled in a timely manner, with responses marked as "Yes" for timely response and "Closed with explanation" for the company\'s response. However, there is at least one complaint regarding a failure to respond to a certified mail complaint, indicating that at least some complaints may not have been handled in a timely manner. \n\nSpecifically, the complaint with complaint ID 13331376 from Nelnet, Inc. (IN), regarding a transfer and misconduct, states that despite multiple letters sent via Certified Mail, Nelnet never responded to the CM, nor provided answers. This suggests that this particular complaint was not handled in a timely manner.\n\nTherefore, the answer is: **Yes, some complaints did not get handled in a timely manner.**'

In [51]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans primarily due to issues such as receiving bad or unclear information from their lenders or servicers, difficulties with the repayment process, and alleged malpractices by loan servicing companies. For example, some borrowers experienced problems with incorrect account information, mismanagement of their payment plans, or delays and errors in payment processing. Others faced challenges related to misreported loan statuses, such as loans reported as delinquent or in default despite the borrowers' claims of timely payments. Additionally, some borrowers have raised concerns about illegal or improper reporting of their debts, such as attempts to collect voided or legally questionable debts, or data breaches that compromised their personal information. These issues contribute to difficulties in managing and repaying loans, leading to missed payments and default."

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer:

If the sentences are short and repetitive, the semantic chunker may have a hard time figuring out the boundaries between chunks, and end up lumping too many of them together, making it hard to retrieve a chunk for a specific question, given that it cotains several other questions.

We might also end up with many chunks that are almost identical, if our sentences are very reptitive. 


# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

# ✅ Activity #1

## Generate Synthetic Data

In [62]:
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

In [53]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

In [54]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

In [55]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 20, relationships: 0)

In [56]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '160f05'. Skipping!
Property 'summary' already exists in node '8578f3'. Skipping!
Property 'summary' already exists in node 'f83cdb'. Skipping!
Property 'summary' already exists in node '325d38'. Skipping!
Property 'summary' already exists in node 'e9df18'. Skipping!
Property 'summary' already exists in node '1a9233'. Skipping!
Property 'summary' already exists in node '0398d4'. Skipping!
Property 'summary' already exists in node 'c76666'. Skipping!
Property 'summary' already exists in node '945467'. Skipping!
Property 'summary' already exists in node '8e2a15'. Skipping!
Property 'summary' already exists in node '061371'. Skipping!
Property 'summary' already exists in node 'af1bc7'. Skipping!
Property 'summary' already exists in node 'b6bfe2'. Skipping!
Property 'summary' already exists in node '02d837'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/6 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/43 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '160f05'. Skipping!
Property 'summary_embedding' already exists in node '1a9233'. Skipping!
Property 'summary_embedding' already exists in node '945467'. Skipping!
Property 'summary_embedding' already exists in node '8578f3'. Skipping!
Property 'summary_embedding' already exists in node 'af1bc7'. Skipping!
Property 'summary_embedding' already exists in node 'e9df18'. Skipping!
Property 'summary_embedding' already exists in node '8e2a15'. Skipping!
Property 'summary_embedding' already exists in node 'c76666'. Skipping!
Property 'summary_embedding' already exists in node '0398d4'. Skipping!
Property 'summary_embedding' already exists in node 'b6bfe2'. Skipping!
Property 'summary_embedding' already exists in node 'f83cdb'. Skipping!
Property 'summary_embedding' already exists in node '02d837'. Skipping!
Property 'summary_embedding' already exists in node '325d38'. Skipping!
Property 'summary_embedding' already exists in node '061371'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 40, relationships: 478)

In [57]:
kg.save("loan_data_kg.json")
loan_data_kg = KnowledgeGraph.load("loan_data_kg.json")
loan_data_kg

KnowledgeGraph(nodes: 40, relationships: 478)

In [58]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=loan_data_kg)

In [59]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

### We have our test set! 👇🏼

In [60]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,Department what is it for?,"[Chapter 1 Academic Years, Academic Calendars,...",The context does not explicitly define 'Depart...,single_hop_specifc_query_synthesizer
1,What is 34 CFR 668.3(a) about in student aid r...,[Regulatory Citations Academic year minimums: ...,Regulatory Citations Academic year minimums: 3...,single_hop_specifc_query_synthesizer
2,Wha Chapter 3 is?,[Inclusion of Clinical Work in a Standard Term...,Inclusion of Clinical Work in a Standard Term ...,single_hop_specifc_query_synthesizer
3,How are non-term characteristics defined in re...,[Non-Term Characteristics A program that measu...,A program that measures progress in credit-hou...,single_hop_specifc_query_synthesizer
4,What is Appendix A in relation to disbursement...,[both the credit or clock hours and the weeks ...,Appendix A provides examples illustrating how ...,single_hop_specifc_query_synthesizer
5,How does the proration of Direct Loan limits r...,[<1-hop>\n\nboth the credit or clock hours and...,The proration of Direct Loan limits occurs whe...,multi_hop_abstract_query_synthesizer
6,how payment periods relate to weeks of instruc...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",The context explains that academic years are d...,multi_hop_abstract_query_synthesizer
7,How does policy compliance for term lengths an...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The context explains that clinical work includ...,multi_hop_abstract_query_synthesizer
8,How do Volume 8 and Volume 7 relate to disburs...,[<1-hop>\n\nboth the credit or clock hours and...,Volume 8 discusses the impact of accelerated p...,multi_hop_specific_query_synthesizer
9,Wht is Volume 2 and Volume 7 in the context of...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",Volume 2 discusses the requirements for academ...,multi_hop_specific_query_synthesizer


### Port our test set to LangSmith

In [ ]:
from langsmith import Client

client = Client(api_key=os.environ["LANGCHAIN_API_KEY"])

dataset_name = "Loan Synthetic Data july 26"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic dataset generated using RAGAS for retriever evaluation"
)

In [152]:
langsmith_dataset

Dataset(name='Loan Synthetic Data july 26', description='Loan Synthetic Data', data_type=<DataType.kv: 'kv'>, id=UUID('22264967-b43a-40d0-8f63-da7a2affa9fa'), created_at=datetime.datetime(2025, 7, 26, 8, 45, 46, 136440, tzinfo=datetime.timezone.utc), modified_at=datetime.datetime(2025, 7, 26, 8, 45, 46, 136440, tzinfo=datetime.timezone.utc), example_count=11, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None)

In [ ]:
dataset = client.read_dataset(dataset_name="Loan Synthetic Data july 26")

for idx, row in testset.to_pandas().iterrows():
    inputs = {"question": row["user_input"]}
    outputs = {"answer": row["reference"]}  # or just {"reference": ...} depending on your naming

    metadata = {
        "reference_contexts": row["reference_contexts"]  # optional, if useful to keep
    }

    client.create_example(
        dataset_id=dataset.id,
        inputs=inputs,
        outputs=outputs,
        metadata=metadata,
    )


In [142]:
examples = client.list_examples(dataset_id=dataset.id)
# for ex in examples:
#     print(ex.inputs, ex.outputs, ex.metadata)

## Time to run our experiments

In [184]:
def run_chain_on_dataset(chain, dataset_name, retriever_name):
    client.run_on_dataset(
        dataset_name=dataset_name,
        llm_or_chain_factory=chain,
        project_name=retriever_name,
        description=f"Running {retriever_name} on the July 26 dataset",
        max_concurrency=1,
    )

In [ ]:
run_chain_on_dataset(naive_retrieval_chain, "Loan Synthetic Data july 26", "naive_retrieval_chain")
run_chain_on_dataset(bm25_retrieval_chain, "Loan Synthetic Data july 26", "bm25_retrieval_chain")
run_chain_on_dataset(contextual_compression_retrieval_chain, "Loan Synthetic Data july 26", "contextual_compression_retrieval_chain")
run_chain_on_dataset(multi_query_retrieval_chain, "Loan Synthetic Data july 26", "multi_query_retrieval_chain")
run_chain_on_dataset(parent_document_retrieval_chain, "Loan Synthetic Data july 26", "parent_document_retrieval_chain")
run_chain_on_dataset(ensemble_retrieval_chain, "Loan Synthetic Data july 26", "ensemble_retrieval_chain")
run_chain_on_dataset(semantic_retrieval_chain, "Loan Synthetic Data july 26", "semantic_retrieval_chain")

/var/folders/ns/n39s_yzn62zby9b115fkzw2h0000gn/T/ipykernel_65588/1441487027.py:2: LangChainDeprecationWarning: The following arguments are deprecated and will be removed in a future release: dict_keys(['description', 'max_concurrency']).
  client.run_on_dataset(


View the evaluation results for project 'semantic_retrieval_chain' at:
https://smith.langchain.com/o/17336763-e025-4cab-8ce1-f61b7408e302/datasets/22264967-b43a-40d0-8f63-da7a2affa9fa/compare?selectedSessions=cdcc311f-3c2c-4cf6-985d-98b2b368cf28

View all tests for Dataset Loan Synthetic Data july 26 at:
https://smith.langchain.com/o/17336763-e025-4cab-8ce1-f61b7408e302/datasets/22264967-b43a-40d0-8f63-da7a2affa9fa
[------------------------------------------------->] 11/11

### Time to use RAGAS

Now that we have our latency and cost information, we can use our results to evaluate how well the retreival performed

In [ ]:
# Started creating ragas structure, but need to figure out how to get the reference contexts

from langsmith import Client
from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

from ragas import SingleTurnSample

In [241]:
def create_ragas_dataset(testset, chain):
    ragas_samples = []

    for test_row in testset:
        response = chain.invoke({"question" : test_row.eval_sample.user_input})

        ragas_samples.append(
            SingleTurnSample(
            user_input=test_row.eval_sample.user_input,
            response=response['response'].content,
            retrieved_contexts=[context.page_content for context in response["context"]],
            reference=test_row.eval_sample.reference
            )
        )

    return ragas_samples

def evaluate_with_ragas(testset, chain):
    ragas_samples = create_ragas_dataset(testset, chain)
    dataset = EvaluationDataset(samples=ragas_samples)

    result = evaluate(
        dataset=dataset,
        metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
        llm=evaluator_llm,
        run_config=custom_run_config
    )
    return result


In [ ]:
chains = [
    naive_retrieval_chain,
    bm25_retrieval_chain,
    contextual_compression_retrieval_chain,
    multi_query_retrieval_chain,
    parent_document_retrieval_chain,
    ensemble_retrieval_chain,
]

results = []
for chain in chains:
    result = evaluate_with_ragas(testset, chain)
    print(result)
    results.append(result)


Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

{'context_recall': 0.0000, 'faithfulness': 0.4306, 'factual_correctness': 0.3064, 'answer_relevancy': 0.6023, 'context_entity_recall': 0.0130, 'noise_sensitivity_relevant': 0.0620}


Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

{'context_recall': 0.1212, 'faithfulness': 0.5042, 'factual_correctness': 0.2791, 'answer_relevancy': 0.6860, 'context_entity_recall': 0.0000, 'noise_sensitivity_relevant': 0.1419}


Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

...

Exception raised in Job[28]: TimeoutError()
Exception raised in Job[35]: TimeoutError()
Exception raised in Job[41]: TimeoutError()
Exception raised in Job[47]: TimeoutError()
Exception raised in Job[53]: TimeoutError()
Exception raised in Job[65]: TimeoutError()


{'context_recall': 0.0000, 'faithfulness': 0.5765, 'factual_correctness': 0.3955, 'answer_relevancy': 0.5092, 'context_entity_recall': 0.0000, 'noise_sensitivity_relevant': 0.1892}


Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

Exception raised in Job[22]: LLMDidNotFinishException(The LLM generation was not completed. Please increase try increasing the max_tokens and try again.)
Exception raised in Job[16]: LLMDidNotFinishException(The LLM generation was not completed. Please increase try increasing the max_tokens and try again.)
Exception raised in Job[5]: TimeoutError()
Exception raised in Job[11]: TimeoutError()
Exception raised in Job[14]: TimeoutError()
Exception raised in Job[17]: TimeoutError()
Exception raised in Job[23]: TimeoutError()
Exception raised in Job[26]: TimeoutError()
Exception raised in Job[28]: TimeoutError()
Exception raised in Job[29]: TimeoutError()
Exception raised in Job[31]: TimeoutError()
Exception raised in Job[32]: TimeoutError()
Exception raised in Job[34]: TimeoutError()
Exception raised in Job[35]: TimeoutError()
Exception raised in Job[37]: TimeoutError()
Exception raised in Job[38]: TimeoutError()
Exception raised in Job[40]: TimeoutError()
Exception raised in Job[41]: Time

{'context_recall': 0.1818, 'faithfulness': 0.7504, 'factual_correctness': 0.2886, 'answer_relevancy': 0.6029, 'context_entity_recall': 0.0000, 'noise_sensitivity_relevant': nan}


## Retriever Comparison Summary

![latency and cost results](latency_cost_evals.png)

### 1. Naive Retriever
- Context Recall: 0.2273
- Faithfulness: 0.6560
- Factual Correctness: 0.3709
- Answer Relevancy: 0.5149
- Context Entity Recall: 0.0185
- Noise Sensitivity: 0.1812
- Latency: 3.019s
- Tokens: 83,432
- Cost: $0.0081

2. BM25 Retriever
- Context Recall: 0.0000
- Faithfulness: 0.4306
- Factual Correctness: 0.3064
- Answer Relevancy: 0.6023
- Context Entity Recall: 0.0130
- Noise Sensitivity: 0.0620
- Latency: 1.581s
- Tokens: 37,739
- Cost: $0.0040

3. Contextual Compression Retriever
- Context Recall: 0.1212
- Faithfulness: 0.5042
- Factual Correctness: 0.2791
- Answer Relevancy: 0.6860
- Context Entity Recall: 0.0000
- Noise Sensitivity: 0.1419
- Latency: 3.479s
- Tokens: 30,603
- Cost: $0.0035

4. Multi-Query Retriever
- Context Recall: 0.0000
- Faithfulness: 0.8377
- Factual Correctness: 0.2880
- Answer Relevancy: 0.6017
- Context Entity Recall: 0.0000
- Noise Sensitivity: NaN
- Latency: 4.809
- Tokens: 158,863.00
- Cost: $0.0167

5. Parent Document Retriever
- Context Recall: 0.0000
- Faithfulness: 0.5765
- Factual Correctness: 0.3955
- Answer Relevancy: 0.5092
- Context Entity Recall: 0.0000
- Noise Sensitivity: 0.1892
- Latency: 2.834s
- Tokens: 55,569
- Cost: $0.0058

6. Ensemble Retriever
- Context Recall: 0.1818
- Faithfulness: 0.7504
- Factual Correctness: 0.2886
- Answer Relevancy: 0.6029
- Context Entity Recall: 0.0000
- Noise Sensitivity: NaN
- Latency: 6.493s
- Tokens: 236,745
- Cost: $0.0247

Based on cost, latency, and performance, the **naive retriever** still appears best suited for this project. It achieves the highest context recall and solid overall balance between faithfulness, factual correctness, and latency, while remaining very affordable. While the multi-query retriever has the highest faithfulness score (0.8377) and decent answer relevancy, it comes at a significantly higher cost and latency: more than 2× the cost and 60% longer latency than naive. The ensemble retriever, though strong in faithfulness, is the most expensive and slowest by far. Other retrievers (BM25, contextual compression, parent document) fall short on key performance metrics, especially context recall and factual correctness. Overall, the naive retriever provides the best trade-off for balanced, cost-effective performance.